In [0]:
# Setup paths and read Bronze Delta table

from pyspark.sql.functions import *
from pyspark.sql.window import Window

bronze_path = "/Volumes/workspace/default/banking-transactions-lakehouse-project-selected-dataset-edition/delta/bronze/transactions"
silver_path = "/Volumes/workspace/default/banking-transactions-lakehouse-project-selected-dataset-edition/delta/silver/transactions"

df = spark.read.format("delta").load(bronze_path)
print(f"Total records: {df.count():,}")
df.printSchema()
df.show(5)

Total records: 116,201
root
 |-- Account_No: string (nullable = true)
 |-- DATE: string (nullable = true)
 |-- TRANSACTION_DETAILS: string (nullable = true)
 |-- CHQ.NO.: integer (nullable = true)
 |-- VALUE_DATE: string (nullable = true)
 |-- WITHDRAWAL_AMT: string (nullable = true)
 |-- DEPOSIT_AMT: string (nullable = true)
 |-- BALANCE_AMT: string (nullable = true)
 |-- .: string (nullable = true)

+-------------+---------+--------------------+-------+----------+--------------+--------------+--------------+---+
|   Account_No|     DATE| TRANSACTION_DETAILS|CHQ.NO.|VALUE_DATE|WITHDRAWAL_AMT|   DEPOSIT_AMT|   BALANCE_AMT|  .|
+-------------+---------+--------------------+-------+----------+--------------+--------------+--------------+---+
|409000611074'|29-Jun-17|TRF FROM  Indiafo...|   NULL| 29-Jun-17|          NULL| 1,000,000.00 | 1,000,000.00 |  .|
|409000611074'| 5-Jul-17|TRF FROM  Indiafo...|   NULL|  5-Jul-17|          NULL| 1,000,000.00 | 2,000,000.00 |  .|
|409000611074'|18-Ju

In [0]:
# Drop CHQ.NO. and invalid column

df = df.drop("CHQ.NO.", ".")
df.columns

['Account_No',
 'DATE',
 'TRANSACTION_DETAILS',
 'VALUE_DATE',
 'WITHDRAWAL_AMT',
 'DEPOSIT_AMT',
 'BALANCE_AMT']

In [0]:
# Rename to clean column names

df = df.withColumnRenamed("Account_No", "account_id") \
       .withColumnRenamed("DATE", "transaction_date") \
       .withColumnRenamed("TRANSACTION_DETAILS", "transaction_details") \
       .withColumnRenamed("VALUE_DATE", "value_date") \
       .withColumnRenamed("WITHDRAWAL_AMT", "withdrawal_amount") \
       .withColumnRenamed("DEPOSIT_AMT", "deposit_amount") \
       .withColumnRenamed("BALANCE_AMT", "balance")

df.columns

['account_id',
 'transaction_date',
 'transaction_details',
 'value_date',
 'withdrawal_amount',
 'deposit_amount',
 'balance']

In [0]:
# Remove commas and convert amounts to double

df = df.withColumn("withdrawal_amount", regexp_replace(col("withdrawal_amount"), ",", "").cast("double")) \
       .withColumn("deposit_amount", regexp_replace(col("deposit_amount"), ",", "").cast("double")) \
       .withColumn("balance", regexp_replace(col("balance"), ",", "").cast("double"))

df.printSchema()
df.select("withdrawal_amount", "deposit_amount", "balance").show(5)

root
 |-- account_id: string (nullable = true)
 |-- transaction_date: string (nullable = true)
 |-- transaction_details: string (nullable = true)
 |-- value_date: string (nullable = true)
 |-- withdrawal_amount: double (nullable = true)
 |-- deposit_amount: double (nullable = true)
 |-- balance: double (nullable = true)

+-----------------+--------------+---------+
|withdrawal_amount|deposit_amount|  balance|
+-----------------+--------------+---------+
|             NULL|     1000000.0|1000000.0|
|             NULL|     1000000.0|2000000.0|
|             NULL|      500000.0|2500000.0|
|             NULL|     3000000.0|5500000.0|
|             NULL|      500000.0|6000000.0|
+-----------------+--------------+---------+
only showing top 5 rows


In [0]:
# Create amount column (deposit if not null, else withdrawal)

df = df.withColumn("amount", 
    when(col("deposit_amount").isNotNull(), col("deposit_amount"))
    .otherwise(col("withdrawal_amount"))
)

df.select("withdrawal_amount", "deposit_amount", "amount").show(10)

+-----------------+--------------+---------+
|withdrawal_amount|deposit_amount|   amount|
+-----------------+--------------+---------+
|             NULL|     1000000.0|1000000.0|
|             NULL|     1000000.0|1000000.0|
|             NULL|      500000.0| 500000.0|
|             NULL|     3000000.0|3000000.0|
|             NULL|      500000.0| 500000.0|
|             NULL|      500000.0| 500000.0|
|             NULL|      500000.0| 500000.0|
|             NULL|      500000.0| 500000.0|
|             NULL|      500000.0| 500000.0|
|             NULL|      500000.0| 500000.0|
+-----------------+--------------+---------+
only showing top 10 rows


In [0]:
# Create transaction_type (credit for deposits, debit for withdrawals)

df = df.withColumn("transaction_type",
    when(col("deposit_amount").isNotNull(), lit("credit"))
    .otherwise(lit("debit"))
)

df.groupBy("transaction_type").count().show()

+----------------+-----+
|transaction_type|count|
+----------------+-----+
|          credit|62652|
|           debit|53549|
+----------------+-----+



In [0]:
# Convert transaction_date to proper date format

df = df.withColumn("transaction_date", to_date(col("transaction_date"), "d-MMM-yy"))
df.select("transaction_date").show(10)

+----------------+
|transaction_date|
+----------------+
|      2017-06-29|
|      2017-07-05|
|      2017-07-18|
|      2017-08-01|
|      2017-08-16|
|      2017-08-16|
|      2017-08-16|
|      2017-08-16|
|      2017-08-16|
|      2017-08-16|
+----------------+
only showing top 10 rows


In [0]:
# Filter out null amounts and remove duplicates

print(f"Before: {df.count():,}")
df = df.filter(col("amount").isNotNull())
print(f"After null filter: {df.count():,}")
df = df.dropDuplicates()
print(f"After deduplication: {df.count():,}")

Before: 116,201
After null filter: 116,201
After deduplication: 116,162


In [0]:
# Check for data quality issues

from datetime import datetime

print("Data Quality Checks:")
print(f"Negative balances: {df.filter(col('balance') < 0).count():,}")
print(f"Negative amounts: {df.filter(col('amount') < 0).count():,}")
print(f"Future dates: {df.filter(col('transaction_date') > datetime.now().date()).count():,}")
print(f"Missing transaction_date: {df.filter(col('transaction_date').isNull()).count():,}")
print(f"Missing account_id: {df.filter(col('account_id').isNull()).count():,}")
print("\nAll checks passed!" if df.filter((col('balance') < 0) | (col('amount') < 0) | (col('transaction_date') > datetime.now().date())).count() == 0 else "\n⚠️ Data quality issues found!")

Data Quality Checks:
Negative balances: 113,237
Negative amounts: 0
Future dates: 0
Missing transaction_date: 0
Missing account_id: 0

⚠️ Data quality issues found!


In [0]:
# Select final columns for Silver layer

df_silver = df.select(
    "account_id",
    "transaction_date",
    "amount",
    "transaction_type",
    "balance"
).orderBy("account_id", col("transaction_date").desc())

df_silver.printSchema()
df_silver.show(20, truncate=False)

root
 |-- account_id: string (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- amount: double (nullable = true)
 |-- transaction_type: string (nullable = false)
 |-- balance: double (nullable = true)

+----------+----------------+---------+----------------+----------------+
|account_id|transaction_date|amount   |transaction_type|balance         |
+----------+----------------+---------+----------------+----------------+
|1196428'  |2019-03-05      |117934.3 |credit          |-1.68782433183E9|
|1196428'  |2019-03-05      |1000000.0|debit           |-1.67884755405E9|
|1196428'  |2019-03-05      |947151.0 |debit           |-1.68529470505E9|
|1196428'  |2019-03-05      |500000.0 |credit          |-1.68974755405E9|
|1196428'  |2019-03-05      |1000000.0|debit           |-1.68134755405E9|
|1196428'  |2019-03-05      |500000.0 |debit           |-1.68434755405E9|
|1196428'  |2019-03-05      |300000.0 |debit           |-1.68722433183E9|
|1196428'  |2019-03-05      |1000000.0|d

In [0]:
# Save as Silver Delta table

df_silver.write.format("delta").mode("overwrite").save(silver_path)
print(f"✓ Saved to: {silver_path}")

✓ Saved to: /Volumes/workspace/default/banking-transactions-lakehouse-project-selected-dataset-edition/delta/silver/transactions


In [0]:
# Display final metrics

print(f"Total Records: {df_silver.count():,}")
print(f"Unique Accounts: {df_silver.select('account_id').distinct().count():,}")
print(f"Date Range: {df_silver.agg(min('transaction_date')).collect()[0][0]} to {df_silver.agg(max('transaction_date')).collect()[0][0]}")
print(f"Credits: {df_silver.filter(col('transaction_type') == 'credit').count():,}")
print(f"Debits: {df_silver.filter(col('transaction_type') == 'debit').count():,}")
print(f"\nTotal Amount: ${df_silver.agg(sum('amount')).collect()[0][0]:,.2f}")
print(f"Average Transaction: ${df_silver.agg(avg('amount')).collect()[0][0]:,.2f}")

Total Records: 116,162
Unique Accounts: 10
Date Range: 2015-01-01 to 2019-03-05
Credits: 62,637
Debits: 53,525

Total Amount: $478,569,847,608.26
Average Transaction: $4,119,848.55
